## Pro-Football Reference Coaches Scrape

### Helper Functions

In [26]:
import os
from bs4 import BeautifulSoup
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time
import re
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import requests

In [27]:
DIR = 'people'
DIR2 = 'teams'

In [28]:
def name(url):
    url = url.replace('-','_')
    return url.split('/')[-1]

In [29]:
def save(link,directory,sleep=10,name=name):
    save_path = os.path.join(directory, name(link))
    if not(os.path.exists(save_path)):
        time.sleep(sleep)
        response = requests.get(link);
        text = response.text
        with open(save_path, "w+") as f:
            f.write(text)
    else :
        with open(save_path, 'r') as f:
            text = f.read()
    return text

In [30]:
def save_tag(link,directory,tag,sleep=10,name=name):
    save_path = os.path.join(directory, name(link))
    if not(os.path.exists(save_path)):
        response = requests.get(link);
        text = response.text
        bs = BeautifulSoup(text, 'html.parser')
        text = bs.find(id = tag)
        time.sleep(sleep)
        with open(save_path, "w+") as f:
            f.write(str(text))
    else :
        with open(save_path, 'r') as f:
            text = f.read()
    return text

In [31]:
def toBS(text):
    bs = BeautifulSoup(text, 'html.parser')
    return bs

In [32]:
def save_tag(link,directory=DIR,tag="content",sleep=10,name=name):
    save_path = os.path.join(directory, name(link))
    if not(os.path.exists(save_path)):
        response = requests.get(link);
        text = response.text
        bs = BeautifulSoup(text, 'html.parser')
        text = bs.find(id = tag)
        time.sleep(sleep)
        with open(save_path, "w+") as f:
            f.write(str(text))
    else :
        with open(save_path, 'r') as f:
            text = f.read()
    return text

### Coach Specific Functions

In [33]:
def extractCoach(coachURL):
    return coachURL.split('/')[-1].split('.')[0]

In [34]:
def getCoach(code):
    return f"https://www.pro-football-reference.com/coaches/{code}.htm"

In [35]:
def nameTeam(url):
    splits = url.split('/')
    return splits[-2] + splits[-1]

In [36]:
def name2(url):
    splits = url.split('/')
    return splits[-2] + splits[-1] + '.html'

In [39]:
def hrefs(arr):
    base = 'https://www.pro-football-reference.com'
    return [base+a['href'] for a in arr]

In [40]:
url = 'https://www.pro-football-reference.com/coaches/'
name2(url)
tag = "content"
bs = toBS(save_tag(url,DIR,tag=tag,name=name2))
a = hrefs(bs.find_all('a'))

In [88]:
def fixCol(df,col,typ):
    df[col] = df[col].fillna(0).astype(typ)
    return df

In [43]:
def resultsFooter(url):
    pass

SyntaxError: 'continue' not properly in loop (922131024.py, line 2)

In [120]:
def resultsPD(url,footerBool=False):
    bs = toBS(save_tag(url))
    df = pd.read_html(str(bs.find(id="all_coaching_results")))[0]
    # df.droplevel(0)
    df.columns = df.columns.get_level_values(1)
    footerRows = df['Year'].apply(lambda x: isinstance(x, str) and not x.isdigit())
    footer = df[footerRows]
    df = df[~footerRows]
    for col in ['Year','G plyf','W plyf','L plyf','Rank']:
        fixCol(df,col,int)
    return df if not footerBool else footer

In [ ]:
print(getCoach('BowlTo0'))
df = resultsPD(getCoach('BowlTo0'))
df18 = df[df['Year'] == 2018]
df

In [169]:
# TODO: need to think about removing interim stints
# could look at team page and see if they are the first coach listed -> scrape would take time though

In [ ]:
def addFeature(tenure=False)

In [270]:
def coachDF(coachLink):
    df = resultsPD(coachLink)
    df.insert(0,'label',df['Year'].apply(lambda x: extractCoach(coachLink)+str(x%100)))
    df.loc[0, 'Tenure'] = 1
    df.loc[0,'Tenure_W'] = df.loc[0,'W']
    df.loc[0,'Tenure_L'] = df.loc[0,'L']
    df.loc[0,'Tenure_PW'] = df.loc[0,'W plyf']
    df.loc[0,'Tenure_PL'] = df.loc[0,'L plyf']
    df.loc[0,'Total_W'] = df.loc[0,'W']
    df.loc[0,'Total_L'] = df.loc[0,'L']
    df.loc[0,'Total_PW'] = df.loc[0,'W plyf']
    df.loc[0,'Total_PL'] = df.loc[0,'L plyf']
    df.loc[0,'Exp'] = 1
    df.loc[0, 'Fired'] = 0
    for i in range(1, len(df)):
        if not df.loc[i, 'Tm'] == df.loc[i-1, 'Tm']:
            df.loc[i, 'Tenure'] = 1
            df.loc[i,'Tenure_W'] = df.loc[i,'W']
            df.loc[i,'Tenure_L'] = df.loc[i,'L']
            df.loc[i,'Tenure_PW'] = df.loc[i,'W plyf']
            df.loc[i,'Tenure_PL'] = df.loc[i,'L plyf']
            df.loc[i-1, 'Fired'] = 1
        else:
            df.loc[i, 'Tenure'] = df.loc[i-1, 'Tenure'] + 1
            df.loc[i, 'Tenure_W'] = df.loc[i-1, 'Tenure_W'] + df.loc[i,'W']
            df.loc[i, 'Tenure_L'] = df.loc[i-1, 'Tenure_L'] + df.loc[i,'L']
            df.loc[i, 'Tenure_PW'] = df.loc[i-1, 'Tenure_PW'] + df.loc[i,'W plyf']
            df.loc[i, 'Tenure_PL'] = df.loc[i-1, 'Tenure_PL'] + df.loc[i,'L plyf']
            df.loc[i-1, 'Fired'] = 0
        df.loc[i,'Total_W'] = df.loc[i-1, 'Total_W']+ df.loc[i,'W']
        df.loc[i,'Total_L'] = df.loc[i-1, 'Total_L']+ df.loc[i,'L']
        df.loc[i,'Total_PW'] = df.loc[i-1, 'Total_PW']+ df.loc[i,'W plyf']
        df.loc[i,'Total_PL'] = df.loc[i-1, 'Total_PL']+ df.loc[i,'L plyf']
        df.loc[i,'Exp'] = i+1
#     TODO - hardcode coaches who were not fired, but instead retired
    df.loc[len(df)-1,'Fired'] = 1
    return df

In [310]:
# df_list = [coachDF(coach) for coach in a]
# df_list

In [318]:
df_concat = pd.concat(df_list, ignore_index=True)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [321]:
base = coachDF(a[0])
newDF = coachDF(a[1])
base = pd.concat([base,newDF],ignore_index=True)
base = pd.concat([base,newDF])
base

,label,Year,Age,Tm,Lg,G,W,L,T,W-L%,...,Tenure_W,Tenure_L,Tenure_PW,Tenure_PL,Total_W,Total_L,Total_PW,Total_PL,Exp,Fired
0,ShulDo063,1963,33,BAL,NFL,14,8,6,0,0.571,...,8.0,6.0,0.0,0.0,8.0,6.0,0.0,0.0,1.0,0.0
1,ShulDo064,1964,34,BAL,NFL,14,12,2,0,0.857,...,20.0,8.0,0.0,1.0,20.0,8.0,0.0,1.0,2.0,0.0
2,ShulDo065,1965,35,BAL,NFL,14,10,3,1,0.769,...,30.0,11.0,0.0,2.0,30.0,11.0,0.0,2.0,3.0,0.0
3,ShulDo066,1966,36,BAL,NFL,14,9,5,0,0.643,...,39.0,16.0,0.0,2.0,39.0,16.0,0.0,2.0,4.0,0.0
4,ShulDo067,1967,37,BAL,NFL,14,11,1,2,0.917,...,50.0,17.0,0.0,2.0,50.0,17.0,0.0,2.0,5.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,HalaGe063,1963,68,CHI,NFL,14,11,1,2,0.917,...,292.0,121.0,6.0,3.0,292.0,121.0,6.0,3.0,36.0,0.0
36,HalaGe064,1964,69,CHI,NFL,14,5,9,0,0.357,...,297.0,130.0,6.0,3.0,297.0,130.0,6.0,3.0,37.0,0.0
37,HalaGe065,1965,70,CHI,NFL,14,9,5,0,0.643,...,306.0,135.0,6.0,3.0,306.0,135.0,6.0,3.0,38.0,0.0
38,HalaGe066,1966,71,CHI,NFL,14,5,7,2,0.417,...,311.0,142.0,6.0,3.0,311.0,142.0,6.0,3.0,39.0,0.0


In [369]:
c1 = coachDF(a[0])
c2 = coachDF(a[1])
base = pd.concat([c1,c2],axis=0)
# base.reset_index(inplace=True, drop=True)
c3 = coachDF(a[2])
pd.concat([base,c3],axis=0)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [290]:
base = coachDF(a[0])
for coach in tqdm(range(1,len(a))):
    addDF = coachDF(a[coach])
    base = pd.concat([base,addDF],ignore_index=True)
    print(a[coach])

  0%|                                           | 1/523 [00:00<01:07,  7.70it/s]

https://www.pro-football-reference.com/coaches/HalaGe0.htm


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [370]:
coachDF(a[1])

,label,Year,Age,Tm,Lg,G,W,L,T,W-L%,...,Tenure_W,Tenure_L,Tenure_PW,Tenure_PL,Total_W,Total_L,Total_PW,Total_PL,Exp,Fired
0,HalaGe020,1920,25,CHI,APFA,13,10,1,2,0.909,...,10.0,1.0,0.0,0.0,10.0,1.0,0.0,0.0,1.0,0.0
1,HalaGe021,1921,26,CHI,APFA,11,9,1,1,0.900,...,19.0,2.0,0.0,0.0,19.0,2.0,0.0,0.0,2.0,0.0
2,HalaGe022,1922,27,CHI,NFL,12,9,3,0,0.750,...,28.0,5.0,0.0,0.0,28.0,5.0,0.0,0.0,3.0,0.0
3,HalaGe023,1923,28,CHI,NFL,12,9,2,1,0.818,...,37.0,7.0,0.0,0.0,37.0,7.0,0.0,0.0,4.0,0.0
4,HalaGe024,1924,29,CHI,NFL,11,6,1,4,0.857,...,43.0,8.0,0.0,0.0,43.0,8.0,0.0,0.0,5.0,0.0
5,HalaGe025,1925,30,CHI,NFL,17,9,5,3,0.643,...,52.0,13.0,0.0,0.0,52.0,13.0,0.0,0.0,6.0,0.0
6,HalaGe026,1926,31,CHI,NFL,16,12,1,3,0.923,...,64.0,14.0,0.0,0.0,64.0,14.0,0.0,0.0,7.0,0.0
7,HalaGe027,1927,32,CHI,NFL,14,9,3,2,0.750,...,73.0,17.0,0.0,0.0,73.0,17.0,0.0,0.0,8.0,0.0
8,HalaGe028,1928,33,CHI,NFL,13,7,5,1,0.583,...,80.0,22.0,0.0,0.0,80.0,22.0,0.0,0.0,9.0,0.0
9,HalaGe029,1929,34,CHI,NFL,15,4,9,2,0.308,...,84.0,31.0,0.0,0.0,84.0,31.0,0.0,0.0,10.0,0.0


In [197]:
df.to_csv('test1.csv',index=False)

In [190]:
links = toBS(save_tag(a[0])).find(id="all_coaching_results").find_all('a')
teams = [t for t in hrefs(links) if "teams" in t and "_games" not in t]
# save_tag(teams[0],DIR2,'meta',name=nameTeam)

In [207]:
links = toBS(save_tag(i)).find(id="all_coaching_results").find_all('a')
teams = [t for t in hrefs(links) if "teams" in t and "_games" not in t]
int(teams[0].split('/')[-1].split('.')[0])

1952

In [208]:
def getYear(link):
    return int(link.split('/')[-1].split('.')[0])

In [211]:
# Finish later!
for i in tqdm(a):
    save_tag(i)
    links = toBS(save_tag(i)).find(id="all_coaching_results").find_all('a')
    teams = [t for t in hrefs(links) if "teams" in t and "_games" not in t]
    for t in teams:
        yr = getYear(t);
        if yr > 2000:
            save_tag(t,DIR2,'meta',name=nameTeam)

100%|████████████████████████████████████████| 524/524 [00:02<00:00, 219.43it/s]


In [183]:
DIR = 'people'
DIR2 = 'teams'

In [ ]:
url = 'https://www.pro-football-reference.com/coaches/'
save_tag(url,DIR,)